# m19 lab · Motion and drift

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Catheter positions move with breathing and the heartbeat, and impedance coordinates drift over a procedure. You model breathing as an asymmetric periodic motion, trade gating duty cycle against residual motion, decide when a reference catheter helps, correct with a surrogate by regression, estimate drift from repeated visits to a landmark, and measure what averaging costs in delay.

| Part | What you do | Equation |
| --- | --- | --- |
| A | A breathing model | eq-m19-breathing |
| B | Respiratory gating | eq-m19-gating |
| C | Reference catheters and surrogates | eq-m19-reference, eq-m19-surrogate |
| D | Drift from repeated visits | eq-m19-drift |
| E | Averaging and delay | eq-m19-average |

Lengths in mm, times in s. The breathing period and excursion, the correlations and the drift rate are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import motion as MO

## Part A · A breathing model

Breathing is periodic but asymmetric, with more time near exhale (Lujan and colleagues). Model the heart's displacement
from its exhale position as $-b\cos^{2n}(\pi t/T)$ (eq-m19-breathing); larger $n$ flattens the exhale plateau. The 4 s
period and 12 mm excursion are illustrative.

In [ ]:
T, b = 4.0, 12.0
t = np.linspace(0, 2 * T, 2001)
for n in (1, 2, 3):
    plt.plot(t, MO.breathing(t, T, b, n), label=f"n = {n}")
plt.xlabel("time (s)"); plt.ylabel("displacement from exhale (mm)"); plt.legend(); plt.show()
for n in (1, 2, 3):
    z = MO.breathing(np.linspace(0, T, 100000), T, b, n)
    print(f"n = {n}: mean displacement {z.mean():6.2f} mm; time within 1.2 mm of exhale {100 * np.mean(-z <= 1.2):4.1f}%")

In [ ]:
# Checks
zs = [MO.breathing(np.linspace(0, T, 100000), T, b, n).mean() for n in (1, 2, 3)]
assert zs[0] < zs[1] < zs[2] < 0

## Part B · Respiratory gating

Accept a point only while the displacement is within $\delta$ of exhale. The accepted fraction of time is
$1 - (2/\pi)\arccos((\delta/b)^{1/(2n)})$ (eq-m19-gating), and the motion that remains is at most $\delta$.

In [ ]:
for n in (1, 2, 3):
    for w in (1.0, 2.0, 4.0):
        duty, rms = MO.gating_numeric(T, b, w, n)
        print(f"n = {n}, window {w:3.1f} mm: duty {100 * duty:5.1f}% (formula {100 * MO.gating_duty_cycle(w, b, n):5.1f}%), RMS residual {rms:4.2f} mm")

In [ ]:
# Checks
d, r = MO.gating_numeric(T, b, 2.0, 2)
assert abs(d - MO.gating_duty_cycle(2.0, b, 2)) < 1e-4 and r < 2.0

## Part C · Reference catheters and surrogates

A reference catheter, or any surrogate of the heart's motion, can be subtracted: the residual SD is
$\sqrt{\sigma_x^2 + \sigma_r^2 - 2\rho\sigma_x\sigma_r}$, which beats doing nothing only when $\rho > \sigma_r/(2\sigma_x)$
(eq-m19-reference). Scaling the surrogate by the regression slope first leaves $\sigma_x\sqrt{1 - \rho^2}$
(eq-m19-surrogate). In one MR study the slope relating coronary to diaphragm motion averaged about 0.6 but ranged from
0.17 to 0.93 between people (Danias and colleagues): slopes must be patient-specific.

In [ ]:
rng = np.random.default_rng(19)
n = 20000
for slope, noise in ((0.6, 1.0), (0.2, 2.0), (0.9, 0.5)):
    diaphragm = 10 * rng.standard_normal(n)
    heart = slope * diaphragm + noise * rng.standard_normal(n)
    k, c = MO.fit_slope(diaphragm[:200], heart[:200])           # fit on a short calibration run
    rho = np.corrcoef(diaphragm, heart)[0, 1]
    print(f"true slope {slope}: fitted {k:.2f}; SD uncorrected {heart.std():.2f}, subtract raw {np.std(heart - diaphragm):.2f}, "
          f"regress {np.std(heart - (k * diaphragm + c)):.2f} (theory {MO.residual_sd(heart.std(), rho):.2f}) mm")

In [ ]:
# Checks
assert MO.reference_helps(1.0, 1.0, 0.6) and not MO.reference_helps(1.0, 1.0, 0.4)

## Part D · Drift from repeated visits

Impedance coordinates drift. Revisit a landmark now and then; the least-squares slope estimates the drift, with
standard error $\sigma/\sqrt{\sum(t_i - \bar t)^2}$ (eq-m19-drift). One study found a landmark reproducible within
1.4 ± 1.1 mm after 2 hours (Wittkampf and colleagues). The drift rate and noise here are illustrative.

In [ ]:
t = np.arange(0, 121, 15.0)                 # minutes
x = 0.02 * t + 0.8 * rng.standard_normal(t.size)
slope, se = MO.drift_fit(t, x)
print(f"estimated drift {slope * 60:.2f} ± {se * 60:.2f} mm per hour (true 1.20); predicted SE {MO.drift_se(t, 0.8) * 60:.2f}")
for span in (30, 60, 120):
    tt = np.linspace(0, span, 5)
    print(f"5 visits over {span:3d} min: SE {MO.drift_se(tt, 0.8) * 60:.2f} mm per hour")

In [ ]:
# Checks
assert MO.drift_se(np.linspace(0, 120, 5), 0.8) < MO.drift_se(np.linspace(0, 30, 5), 0.8)

## Part E · Averaging and delay

Averaging position over a window $W$ attenuates motion of frequency $f$ by $\lvert\operatorname{sinc}(fW)\rvert$ and delays
it by $W/2$ (eq-m19-average). A window of exactly one breath removes breathing but lags by half a breath.

In [ ]:
fs = 100.0
tt = np.arange(0, 20, 1 / fs)
z = MO.breathing(tt, T, b, 1) + b / 2                    # zero-mean for n = 1
for W in (0.5, 2.0, 4.0):
    y = MO.moving_average(z, int(W * fs))
    print(f"window {W:3.1f} s: gain at the breathing frequency {MO.moving_average_gain(1 / T, W):.3f}, "
          f"residual {np.std(y[int(8 * fs):]):.3f} mm, delay {W / 2:.2f} s")

In [ ]:
# Checks
y = MO.moving_average(z, int(T * fs))          # one full breath
assert np.std(y[int(8 * fs):]) < 1e-9

## Where to go next

- Gate on the ECG as well as on breathing, and see how the two duty cycles multiply.
- Fit the drift with a smooth curve instead of a line and compare.
- m20 builds geometry from the gated, corrected positions.